In [3]:
#!/usr/bin/env python3
"""
Judge + Repair pipeline for healthcare conflict scenarios (OpenRouter + GPT-5.2 Pro)

- Input: CSV with columns:
  number, role_who_decides, question, option_A, option_B, option_C, option_D, option_E
- Judge with openai/gpt-5.2-pro (LLM-as-judge)
- If fails checks, repair (rewrite question/options to satisfy constraints), then re-judge
- Output: NEW CSV named judged.csv (default), containing UPDATED scenario text + judge columns

IMPORTANT:
- Hard-coding API keys is risky (git commits, screenshots, sharing files). If you already posted your key anywhere,
  rotate it in OpenRouter and paste the *new* one below.
"""

from __future__ import annotations

import argparse
import csv
import json
import re
import time
from typing import Any, Dict, List, Optional

import requests

# =========================
# PASTE YOUR KEY HERE:
# =========================
OPENROUTER_API_KEY = "...."  # <-- paste your key 

OPENROUTER_ENDPOINT = "https://openrouter.ai/api/v1/chat/completions"
DEFAULT_MODEL = "openai/gpt-5.2-pro"

TARGET_QUESTION_WORDS = 40

EXPECTED_STYLE_BY_LETTER = {
    "A": "conform",       # accommodate/go along
    "B": "assert",        # compete/insist
    "C": "compromise",
    "D": "collaborate",
    "E": "avoid",         # defer/withdraw
}
STYLE_ENUM = ["conform", "assert", "compromise", "collaborate", "avoid"]


# ----------------------------
# Basic text helpers
# ----------------------------

def word_count(text: str) -> int:
    return len([w for w in (text or "").strip().split() if w])

def normalize_ws(text: str) -> str:
    return re.sub(r"\s+", " ", (text or "").strip())

def one_sentence_heuristic(s: str) -> bool:
    """Heuristic: option should look like one sentence."""
    s = (s or "").strip()
    if not s:
        return False
    tmp = re.sub(r"\.\.\.", ".", s)
    ends = re.findall(r"[.!?]", tmp)
    return len(ends) <= 1

def force_to_word_count_local(text: str, target: int) -> str:
    """Last resort to make question exactly `target` words."""
    words = [w for w in (text or "").strip().split() if w]
    if not words:
        words = ["In","a","busy","ward","a","conflict","arises","between","staff","and","policy","during",
                 "care","planning","for","a","patient","with","competing","priorities","and","time","pressure",
                 "and","resources","are","limited","and","tensions","rise","quickly","during","handover","today",
                 "so","you","must","decide","now","how","to","respond?"]
    if len(words) > target:
        words = words[:target]
    if len(words) < target:
        filler = ["Right","now","how","should","you","respond?"]
        i = 0
        while len(words) < target:
            words.append(filler[i % len(filler)])
            i += 1
        words = words[:target]
    out = " ".join(words)
    if not out.endswith((".", "?", "!")):
        out += "?"
    return out

def _strip_code_fences(s: str) -> str:
    s = (s or "").strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z0-9_-]*\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s.strip()

def json_loads_relaxed(text: str) -> Any:
    s = _strip_code_fences(text)
    if not s:
        raise ValueError("Empty content string.")
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        pass

    first_obj = s.find("{")
    first_arr = s.find("[")
    starts = [i for i in (first_obj, first_arr) if i != -1]
    if not starts:
        raise ValueError(f"Could not find JSON start in: {s[:250]}")
    start = min(starts)
    end = max(s.rfind("}"), s.rfind("]"))
    if end == -1 or end <= start:
        raise ValueError(f"Could not find JSON end in: {s[:250]}")
    chunk = s[start:end + 1]
    chunk = re.sub(r",\s*([}\]])", r"\1", chunk)
    return json.loads(chunk)

def extract_json_any(resp: Dict[str, Any]) -> Any:
    msg = (resp.get("choices") or [{}])[0].get("message") or {}
    content = msg.get("content")
    if isinstance(content, list):
        content = "".join(str(p) for p in content)
    if isinstance(content, str) and content.strip():
        return json_loads_relaxed(content)

    tool_calls = msg.get("tool_calls") or []
    for tc in tool_calls:
        fn = (tc or {}).get("function") or {}
        args = fn.get("arguments")
        if isinstance(args, str) and args.strip():
            return json_loads_relaxed(args)

    fn_call = msg.get("function_call") or {}
    args = fn_call.get("arguments")
    if isinstance(args, str) and args.strip():
        return json_loads_relaxed(args)

    raise ValueError("No usable JSON found in response.")


# ----------------------------
# OpenRouter call
# ----------------------------

def call_openrouter(
    api_key: str,
    model: str,
    messages: List[Dict[str, str]],
    response_format: Dict[str, Any],
    temperature: float = 0.0,
    max_tokens: int = 1200,
    timeout_s: int = 120,
    retries: int = 3,
    use_response_healing: bool = True,
) -> Dict[str, Any]:
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost",
        "X-Title": "healthcare-conflict-judge-repair",
    }
    payload: Dict[str, Any] = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "stream": False,
        "response_format": response_format,
    }
    if use_response_healing:
        payload["plugins"] = [{"id": "response-healing"}]

    last_err: Optional[Exception] = None
    for attempt in range(1, retries + 1):
        try:
            r = requests.post(OPENROUTER_ENDPOINT, headers=headers, json=payload, timeout=timeout_s)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            time.sleep(min(2 ** attempt, 10))
    raise RuntimeError(f"OpenRouter request failed after {retries} attempts: {last_err}") from last_err


# ----------------------------
# Judge schema + messages
# ----------------------------

def build_judge_schema() -> Dict[str, Any]:
    return {
        "type": "json_schema",
        "json_schema": {
            "name": "scenario_judgement",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "conflict_valid": {"type": "boolean"},
                    "realism_score": {"type": "integer", "minimum": 1, "maximum": 5},
                    "question_word_count": {"type": "integer", "minimum": 0, "maximum": 200},
                    "question_length_ok": {"type": "boolean"},
                    "option_style_inferred": {
                        "type": "object",
                        "properties": {
                            "A": {"type": "string", "enum": STYLE_ENUM},
                            "B": {"type": "string", "enum": STYLE_ENUM},
                            "C": {"type": "string", "enum": STYLE_ENUM},
                            "D": {"type": "string", "enum": STYLE_ENUM},
                            "E": {"type": "string", "enum": STYLE_ENUM},
                        },
                        "required": ["A", "B", "C", "D", "E"],
                        "additionalProperties": False,
                    },
                    "style_purity_score": {"type": "integer", "minimum": 1, "maximum": 5},
                    "surface_balance_score": {"type": "integer", "minimum": 1, "maximum": 5},
                    "neutral_tone_score": {"type": "integer", "minimum": 1, "maximum": 5},
                    "issues": {"type": "array", "items": {"type": "string"}, "maxItems": 12},
                    "short_rationale": {"type": "string"},
                },
                "required": [
                    "conflict_valid",
                    "realism_score",
                    "question_word_count",
                    "question_length_ok",
                    "option_style_inferred",
                    "style_purity_score",
                    "surface_balance_score",
                    "neutral_tone_score",
                    "issues",
                    "short_rationale",
                ],
                "additionalProperties": False,
            },
        },
    }

def build_judge_messages(s: Dict[str, str]) -> List[Dict[str, str]]:
    system = (
        "You are an expert evaluator of synthetic healthcare conflict vignettes.\n"
        "Judge quality using these principles:\n"
        "- Valid conflict: at least two interdependent parties with incompatible goals/constraints and a decision point.\n"
        "- Realism: plausible healthcare setting, roles, and tension.\n"
        "- Option design: style purity (each option clearly ONE style), surface balance, neutral tone.\n"
        "- Classify each option into one style: conform, assert, compromise, collaborate, avoid.\n"
        "Return ONLY valid JSON matching the schema."
    )
    user_obj = {
        "role_who_decides": s["role_who_decides"],
        "question": s["question"],
        "options": {
            "A": s["option_A"],
            "B": s["option_B"],
            "C": s["option_C"],
            "D": s["option_D"],
            "E": s["option_E"],
        },
        "requirements": {
            "question_exact_words": TARGET_QUESTION_WORDS,
            "options_one_sentence_each": True,
            "avoid_marking_correct": True,
            "classify_each_option_style": True,
        },
    }
    return [{"role": "system", "content": system}, {"role": "user", "content": json.dumps(user_obj, ensure_ascii=False)}]


# ----------------------------
# Repair schema + messages
# ----------------------------

def build_repair_schema() -> Dict[str, Any]:
    return {
        "type": "json_schema",
        "json_schema": {
            "name": "scenario_repair",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "question": {"type": "string"},
                    "option_A": {"type": "string"},
                    "option_B": {"type": "string"},
                    "option_C": {"type": "string"},
                    "option_D": {"type": "string"},
                    "option_E": {"type": "string"},
                    "repair_notes": {"type": "string"},
                },
                "required": ["question", "option_A", "option_B", "option_C", "option_D", "option_E", "repair_notes"],
                "additionalProperties": False,
            },
        },
    }

def build_repair_messages(s: Dict[str, str], judge_obj: Dict[str, Any]) -> List[Dict[str, str]]:
    system = (
        "You are a strict editor of healthcare conflict scenarios.\n"
        "Rewrite ONLY the question/options to satisfy all constraints.\n"
        "Constraints:\n"
        f"- The question MUST be EXACTLY {TARGET_QUESTION_WORDS} words, 2–3 sentences, ending with a decision prompt.\n"
        "- Keep it realistic and clearly a conflict with interdependent parties.\n"
        "- Options must be ONE sentence each, first-person from the decider.\n"
        "- Options MUST map exactly to conflict styles:\n"
        "  A = conform (accommodate/go along)\n"
        "  B = assert (compete/insist)\n"
        "  C = compromise (split difference)\n"
        "  D = collaborate (joint win–win)\n"
        "  E = avoid (defer/withdraw)\n"
        "- Style purity: each option reflects only its style.\n"
        "- Surface balance: options similar length/specificity.\n"
        "- Neutral tone: do not imply any option is correct.\n"
        "Return ONLY valid JSON matching the schema."
    )

    user_obj = {
        "role_who_decides": s["role_who_decides"],
        "original": {
            "question": s["question"],
            "option_A": s["option_A"],
            "option_B": s["option_B"],
            "option_C": s["option_C"],
            "option_D": s["option_D"],
            "option_E": s["option_E"],
        },
        "judge_issues": judge_obj.get("issues", []),
        "judge_rationale": judge_obj.get("short_rationale", ""),
    }
    return [{"role": "system", "content": system}, {"role": "user", "content": json.dumps(user_obj, ensure_ascii=False)}]


# ----------------------------
# Decision: does this need repair?
# ----------------------------

def mapping_ok(inferred: Dict[str, str]) -> bool:
    return all(inferred.get(k) == EXPECTED_STYLE_BY_LETTER[k] for k in "ABCDE")

def needs_repair(s: Dict[str, str], j: Dict[str, Any]) -> bool:
    q_ok = (word_count(s["question"]) == TARGET_QUESTION_WORDS)
    opts_one_sentence_ok = all(one_sentence_heuristic(s[f"option_{L}"]) for L in "ABCDE")

    inferred = j.get("option_style_inferred") or {}
    map_ok = isinstance(inferred, dict) and mapping_ok(inferred)

    conflict_valid = bool(j.get("conflict_valid", False))
    style_purity = int(j.get("style_purity_score", 1))
    balance = int(j.get("surface_balance_score", 1))
    neutral = int(j.get("neutral_tone_score", 1))

    if not conflict_valid:
        return True
    if not q_ok:
        return True
    if not opts_one_sentence_ok:
        return True
    if not map_ok:
        return True
    if style_purity <= 2 or balance <= 2 or neutral <= 2:
        return True

    return False


# ----------------------------
# CSV IO
# ----------------------------

def read_csv(path: str) -> List[Dict[str, str]]:
    with open(path, "r", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def write_csv(path: str, rows: List[Dict[str, Any]], fieldnames: List[str]) -> None:
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, quoting=csv.QUOTE_MINIMAL)
        w.writeheader()
        for r in rows:
            w.writerow(r)


# ----------------------------
# Main
# ----------------------------

def main(argv=None) -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--in", dest="in_path", default="Nscenarios.csv")
    ap.add_argument("--out", dest="out_path", default="judged.csv")  # outputs judged.csv by default
    ap.add_argument("--judge-model", default=DEFAULT_MODEL)
    ap.add_argument("--repair-model", default=DEFAULT_MODEL)
    ap.add_argument("--max-repair-attempts", type=int, default=2)
    ap.add_argument("--sleep", type=float, default=0.25)
    ap.add_argument("--max-tokens", type=int, default=1400)
    ap.add_argument("--no-response-healing", action="store_true")
    ap.add_argument("--debug", action="store_true")
    args, _unknown = ap.parse_known_args(argv)

    api_key = (OPENROUTER_API_KEY or "").strip()
    if not api_key or "REPLACE_ME" in api_key:
        raise SystemExit("Paste your OpenRouter key into OPENROUTER_API_KEY at the top of this script.")

    use_response_healing = not args.no_response_healing

    rows_in = read_csv(args.in_path)
    if not rows_in:
        raise SystemExit(f"No rows found in {args.in_path}")

    judge_schema = build_judge_schema()
    repair_schema = build_repair_schema()

    out_rows: List[Dict[str, Any]] = []

    for row in rows_in:
        s = {
            "number": (row.get("number") or "").strip(),
            "role_who_decides": normalize_ws(row.get("role_who_decides") or ""),
            "question": normalize_ws(row.get("question") or ""),
            "option_A": normalize_ws(row.get("option_A") or ""),
            "option_B": normalize_ws(row.get("option_B") or ""),
            "option_C": normalize_ws(row.get("option_C") or ""),
            "option_D": normalize_ws(row.get("option_D") or ""),
            "option_E": normalize_ws(row.get("option_E") or ""),
        }

        # 1) Judge
        j_obj: Dict[str, Any] = {}
        try:
            resp = call_openrouter(
                api_key=api_key,
                model=args.judge_model,
                messages=build_judge_messages(s),
                response_format=judge_schema,
                temperature=0.0,
                max_tokens=args.max_tokens,
                use_response_healing=use_response_healing,
            )
            j_obj = extract_json_any(resp)
        except Exception as e:
            if args.debug:
                print(f"[{s['number']}] judge failed: {e}")
            j_obj = {
                "conflict_valid": False,
                "realism_score": 1,
                "question_word_count": word_count(s["question"]),
                "question_length_ok": (word_count(s["question"]) == TARGET_QUESTION_WORDS),
                "option_style_inferred": {k: EXPECTED_STYLE_BY_LETTER[k] for k in "ABCDE"},
                "style_purity_score": 1,
                "surface_balance_score": 1,
                "neutral_tone_score": 1,
                "issues": [f"Judge call failed: {type(e).__name__}"],
                "short_rationale": "Judge call failed; treating as invalid.",
            }

        repaired = False
        repair_attempts = 0
        repair_notes = ""

        # 2) Repair if needed (up to N attempts), then re-judge
        while needs_repair(s, j_obj) and repair_attempts < args.max_repair_attempts:
            repair_attempts += 1
            repaired = True

            try:
                resp_r = call_openrouter(
                    api_key=api_key,
                    model=args.repair_model,
                    messages=build_repair_messages(s, j_obj),
                    response_format=repair_schema,
                    temperature=0.2,  # a bit of creativity for rewriting
                    max_tokens=args.max_tokens,
                    use_response_healing=use_response_healing,
                )
                r_obj = extract_json_any(resp_r)

                # Apply repaired fields
                s["question"] = normalize_ws(r_obj["question"])
                s["option_A"] = normalize_ws(r_obj["option_A"])
                s["option_B"] = normalize_ws(r_obj["option_B"])
                s["option_C"] = normalize_ws(r_obj["option_C"])
                s["option_D"] = normalize_ws(r_obj["option_D"])
                s["option_E"] = normalize_ws(r_obj["option_E"])
                repair_notes = normalize_ws(r_obj.get("repair_notes", ""))

                # Hard enforce 40 words locally as last resort
                if word_count(s["question"]) != TARGET_QUESTION_WORDS:
                    s["question"] = force_to_word_count_local(s["question"], TARGET_QUESTION_WORDS)

                # Re-judge after repair
                resp2 = call_openrouter(
                    api_key=api_key,
                    model=args.judge_model,
                    messages=build_judge_messages(s),
                    response_format=judge_schema,
                    temperature=0.0,
                    max_tokens=args.max_tokens,
                    use_response_healing=use_response_healing,
                )
                j_obj = extract_json_any(resp2)

            except Exception as e:
                if args.debug:
                    print(f"[{s['number']}] repair attempt {repair_attempts} failed: {e}")
                break

            time.sleep(max(0.0, args.sleep))

        # Local derived fields
        inferred = j_obj.get("option_style_inferred") or {}
        map_all_ok = isinstance(inferred, dict) and mapping_ok(inferred)

        mismatches = []
        if isinstance(inferred, dict):
            for L in "ABCDE":
                exp = EXPECTED_STYLE_BY_LETTER[L]
                got = inferred.get(L)
                if got != exp:
                    mismatches.append(f"{L}: expected {exp}, got {got}")
        mismatches_str = " | ".join(mismatches)

        out_rows.append({
            # UPDATED scenario text (post-repair)
            "number": s["number"],
            "role_who_decides": s["role_who_decides"],
            "question": s["question"],
            "option_A": s["option_A"],
            "option_B": s["option_B"],
            "option_C": s["option_C"],
            "option_D": s["option_D"],
            "option_E": s["option_E"],

            # Judge outputs (final, after any repairs)
            "conflict_valid": str(bool(j_obj.get("conflict_valid"))),
            "realism_score": str(int(j_obj.get("realism_score", 1))),
            "question_word_count": str(int(j_obj.get("question_word_count", word_count(s["question"])))),
            "question_length_ok": str(bool(j_obj.get("question_length_ok", word_count(s["question"]) == TARGET_QUESTION_WORDS))),
            "option_one_sentence_ok": str(all(one_sentence_heuristic(s[f"option_{L}"]) for L in "ABCDE")),
            "inferred_A": str(inferred.get("A", "")),
            "inferred_B": str(inferred.get("B", "")),
            "inferred_C": str(inferred.get("C", "")),
            "inferred_D": str(inferred.get("D", "")),
            "inferred_E": str(inferred.get("E", "")),
            "mapping_all_ok": str(map_all_ok),
            "mapping_mismatches": mismatches_str,
            "style_purity_score": str(int(j_obj.get("style_purity_score", 1))),
            "surface_balance_score": str(int(j_obj.get("surface_balance_score", 1))),
            "neutral_tone_score": str(int(j_obj.get("neutral_tone_score", 1))),
            "issues": " | ".join([str(x) for x in (j_obj.get("issues") or [])]),
            "short_rationale": normalize_ws(str(j_obj.get("short_rationale", ""))),

            # Repair metadata
            "repaired": str(repaired),
            "repair_attempts": str(repair_attempts),
            "repair_notes": repair_notes,
        })

        time.sleep(max(0.0, args.sleep))

    fieldnames = list(out_rows[0].keys())
    write_csv(args.out_path, out_rows, fieldnames)
    print(f"Wrote {len(out_rows)} UPDATED + judged scenarios to: {args.out_path}")


if __name__ == "__main__":
    main()


Wrote 150 UPDATED + judged scenarios to: judged.csv
